<img align="right" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>

# Project Setup

**KSO2 Notebook 1 of 5** · Written by the KSO Team

---

Create a KSO2 **project** and attach a **YOLO-formatted dataset** to it. The project is a folder that owns your experiment tracking (MLflow) and training runs. Datasets and base weights stay where they already live on disk; the project stores references to them.

---
### Prerequisites

- KSO2 installed. **LUMI users:** see [`contrib/lumi/README.md`](../contrib/lumi/README.md). **Local users:** `pip install -e .` from the repo root.
- A YOLO-formatted dataset ready, or skip ahead to use the built-in COCO8 sample for a quick test.

> If you don't have a YOLO dataset yet, run **Notebook 0: Prepare Data** first to build one from raw footage or BIIGLE annotations.

**What you'll do in this notebook:**
1. Create your project (generates `<name>.project.yaml` and the base folders)
2. Attach a YOLO dataset to it (your own `data.yaml`, or the built-in COCO8 sample)
3. *(Optional)* Run offline augmentation on the training set
4. Verify everything is ready for training

**Next:** once complete, continue to **Notebook 2: Train and Evaluate Models**.

---
## Phase 0: Setup

Import the managers and helpers used in this notebook. If you haven't installed KSO2 yet, see the **Prerequisites** above.

In [ ]:
from pathlib import Path
import yaml

from kso import ProjectManager, run_augmentation

proj = ProjectManager()

print(" [OK] KSO2 imported successfully")

---
## Phase 1: Create Project

Choose a project name and a storage location. This creates a project folder containing a `<name>.project.yaml` config, an `mlflow.db` for experiment tracking, and a `runs/` folder for YOLO outputs.

- `PROJECT_NAME` is sanitised (lowercased, non-alphanumeric characters become underscores). The sanitised version becomes your folder name.
- `PROJECT_PATH` is the parent directory.

> Some paths in the YAML will show `None` at this stage. These get filled in as you progress through the notebooks.

In [ ]:
PROJECT_NAME = "my_project_name"  # <-- EDIT, for example "MPA_Monitoring_2026"
PROJECT_PATH = "/path/to/parent/dir"  # <-- EDIT, e.g. "/scratch/project_/your_username"

project = proj.create_project(
    project_name=PROJECT_NAME,
    project_path=PROJECT_PATH,
)

print(f"\n[OK] Project created")
print(f"  Name (sanitised): {project.project_name}")
print(f"  Config YAML:      {project.Config_file_path}")

---
## Phase 2: Attach Dataset

Attach a YOLO-formatted dataset to the project. Point `DATA_YAML` at your `data.yaml`, or set it to `None` to use the built-in COCO8 sample.

The dataset is not copied. The project YAML stores a reference to its `data.yaml`. If you later move the dataset, re-run this cell.

In [ ]:
DATA_YAML = "/path/to/your/data.yaml"  # <-- EDIT
# DATA_YAML = None   # use COCO8 sample instead

proj.add_data(
    project=project,
    # data_type="yolo_dataset",
    data_path=DATA_YAML,
)

print("Dataset attached:", DATA_YAML if DATA_YAML else "COCO8 sample")

---
## Phase 3: Augmentation *(optional)*

Generate flipped and rotated copies of your training images and labels on disk. Useful for very small datasets.

Ultralytics already applies online augmentation during training, so this is a supplement, not a requirement. Skip this phase unless you have a specific reason to pre-augment.
> Each run draws a fresh random sample and writes new `_aug_*` files into `train/`. Repeated runs accumulate, they do not replace.

In [ ]:
RUN_AUGMENTATION = False  # <-- set to True to run

if RUN_AUGMENTATION:
    run_augmentation(
        data_yaml_path=project.data_path["ultralytics_data_path"],
        augment_factor=0.5,  # 0.5 adds ~50% more augmented images on top of originals
        random_seed=42,
    )
    print("\n[OK] Augmentation complete")
else:
    print("Skipped. Set RUN_AUGMENTATION = True above to enable.")

---
## Phase 4: Ready Check

Confirm your dataset is reachable, splits are in place, and print the YAML path you'll pass to Notebook 2.

In [ ]:
# Resolve the dataset path directly from the project object.
data_yaml_path = Path(project.data_path["ultralytics_data_path"])

is_sample = (
    "default_dataset" in str(data_yaml_path) or "coco8" in str(data_yaml_path).lower()
)

if is_sample:
    print(f"Project:      {project.project_name}")
    print(f"Dataset YAML: {data_yaml_path}")
    print("Sample dataset attached. Splits will auto-download at training time.")
else:
    with open(data_yaml_path, "r") as f:
        data_cfg = yaml.safe_load(f)

    # In a YOLO data.yaml, `path` is relative to the YAML file's location.
    path_val = data_cfg.get("path")
    data_root = (
        (data_yaml_path.parent / path_val).resolve()
        if path_val
        else data_yaml_path.parent
    )
    names = data_cfg["names"]
    class_list = list(names.values()) if isinstance(names, dict) else names

    print(f"Project:      {project.project_name}")
    print(f"Dataset YAML: {data_yaml_path}")
    print(f"Classes ({len(class_list)}): {class_list}")
    print()

    IMG_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}
    for split in ["train", "val", "test"]:
        split_rel = data_cfg.get(split)
        if not split_rel:
            print(f"  {split:5s}: not defined")
            continue
        split_dir = (data_root / split_rel).resolve()
        if not split_dir.exists():
            print(f"  {split:5s}: MISSING -- {split_dir}")
            continue
        n = sum(
            1
            for p in split_dir.iterdir()
            if p.is_file() and p.suffix.lower() in IMG_EXTS
        )
        print(f"  {split:5s}: {n:5d} images  ({split_dir})")

print(
    f'\n[OK] Ready. Use this in Notebook 2:\n  YAML_PATH = "{project.Config_file_path}"'
)

---
## Done

Your project folder now contains:

```
<PROJECT_PATH>/<project_name>/
|-- <project_name>.project.yaml   # config, pass this to Notebook 2
|-- mlflow.db                     # experiment tracking (populated during training)
`-- runs/                         # YOLO training outputs (populated during training)
```

Continue to **Notebook 2: Train and Evaluate Models**.